In [14]:
import pandas as pd
import min_features, daily_return
import importlib
importlib.reload(min_features)
importlib.reload(daily_return)


perf_df = pd.read_csv("master_run_results.csv")
perf_df['Date'] = perf_df['test_start']
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 
return_cols = df_daily.columns[df_daily.columns.str.contains("Return_")].to_list()
df_returns = df_daily[['Date'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()

In [46]:
df_returns_2 = df_returns[['Date', 'Return_2']].sort_values(by='Date').copy()
# df has columns: ["Date", "Return_1"] where Return_1 is 0/1 (or False/True)

s = df_returns_2["Return_2"].astype(int)

# identify streak groups (new group whenever value changes)
grp = s.ne(s.shift()).cumsum()

# streak length within each group: 1,2,3,...
streak_len = s.groupby(grp).cumcount() + 1

# positive streaks for 1s, negative for 0s
df_returns_2["streak"] = streak_len.where(s.eq(1), -streak_len)
#df_returns_1.sort_values(by='Date', ascending=False)

perf_cols = ['model', 'acc', 'Date', 'train_years', 'feature_set']
performance_2 = pd.merge(df_returns_2, perf_df[(perf_df['horizon'] == 2) & (perf_df['test_days'] == 1)], on='Date', how='inner')
performance_2

,Date,Return_2,streak,Unnamed: 0,run,model,test_days,bal_acc,acc,sign_acc,...,train_start,train_end,test_start,test_end,month,train_years,horizon_days,n_features,feature_set,horizon
0,2025-01-23,0,-2,454,228,xgboost,1,1.0,1.0,1.0,...,2021-02-18,2025-01-22,2025-01-23,2025-01-23,January,4,2,80,daily,2
1,2025-01-23,0,-2,455,228,random_forest,1,1.0,1.0,1.0,...,2021-02-18,2025-01-22,2025-01-23,2025-01-23,January,4,2,80,daily,2
2,2025-01-23,0,-2,910,228,xgboost,1,1.0,1.0,1.0,...,2019-03-04,2025-01-22,2025-01-23,2025-01-23,January,6,2,80,daily,2
3,2025-01-23,0,-2,911,228,random_forest,1,1.0,1.0,1.0,...,2019-03-04,2025-01-22,2025-01-23,2025-01-23,January,6,2,80,daily,2
4,2025-01-23,0,-2,3190,228,xgboost,1,0.0,0.0,-1.0,...,2021-02-18,2025-01-22,2025-01-23,2025-01-23,January,4,2,87,minute,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,2025-12-19,1,3,3193,1,random_forest,1,1.0,1.0,1.0,...,2020-01-30,2025-12-18,2025-12-19,2025-12-19,December,6,2,87,minute,2
2732,2025-12-19,1,3,5472,1,xgboost,1,1.0,1.0,1.0,...,2022-01-12,2025-12-18,2025-12-19,2025-12-19,December,4,2,167,daily+minute,2
2733,2025-12-19,1,3,5473,1,random_forest,1,1.0,1.0,1.0,...,2022-01-12,2025-12-18,2025-12-19,2025-12-19,December,4,2,167,daily+minute,2
2734,2025-12-19,1,3,5928,1,xgboost,1,1.0,1.0,1.0,...,2020-01-30,2025-12-18,2025-12-19,2025-12-19,December,6,2,167,daily+minute,2


In [47]:
df = performance_2.copy()
# 1) Accuracy by (model, train_years, streak)
acc_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak",
    values="acc",
    aggfunc="mean"
)

# 2) Count by (model, train_years, streak)
cnt_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak",
    values="acc",          # any column works since we're counting rows
    aggfunc="size"
)

# 3) Combine into one wide table with clear column labels
out = pd.concat({"acc": acc_piv, "count": cnt_piv}, axis=1)

# optional: sort columns so streaks go -N ... -1, 1 ... N
out = out.reindex(sorted(out.columns, key=lambda x: (x[1] >= 0, x[1])), axis=1)

In [52]:
df[df['streak'] == -3]

,Date,Return_2,streak,Unnamed: 0,run,model,test_days,bal_acc,acc,sign_acc,...,train_start,train_end,test_start,test_end,month,train_years,horizon_days,n_features,feature_set,horizon
12,2025-01-24,0,-3,452,227,xgboost,1,1.0,1.0,1.0,...,2021-02-19,2025-01-23,2025-01-24,2025-01-24,January,4,2,80,daily,2
13,2025-01-24,0,-3,453,227,random_forest,1,1.0,1.0,1.0,...,2021-02-19,2025-01-23,2025-01-24,2025-01-24,January,4,2,80,daily,2
14,2025-01-24,0,-3,908,227,xgboost,1,1.0,1.0,1.0,...,2019-03-05,2025-01-23,2025-01-24,2025-01-24,January,6,2,80,daily,2
15,2025-01-24,0,-3,909,227,random_forest,1,1.0,1.0,1.0,...,2019-03-05,2025-01-23,2025-01-24,2025-01-24,January,6,2,80,daily,2
16,2025-01-24,0,-3,3188,227,xgboost,1,1.0,1.0,1.0,...,2021-02-19,2025-01-23,2025-01-24,2025-01-24,January,4,2,87,minute,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2671,2025-12-12,0,-3,3203,6,random_forest,1,1.0,1.0,1.0,...,2020-01-23,2025-12-11,2025-12-12,2025-12-12,December,6,2,87,minute,2
2672,2025-12-12,0,-3,5482,6,xgboost,1,0.0,0.0,-1.0,...,2022-01-05,2025-12-11,2025-12-12,2025-12-12,December,4,2,167,daily+minute,2
2673,2025-12-12,0,-3,5483,6,random_forest,1,0.0,0.0,-1.0,...,2022-01-05,2025-12-11,2025-12-12,2025-12-12,December,4,2,167,daily+minute,2
2674,2025-12-12,0,-3,5938,6,xgboost,1,0.0,0.0,-1.0,...,2020-01-23,2025-12-11,2025-12-12,2025-12-12,December,6,2,167,daily+minute,2


In [48]:
gcols = ["model", "train_years", "feature_set"]

df2 = df.copy()
df2["side"] = df2["streak"].gt(0).map({True: "pos", False: "neg"})  # 0 shouldn't exist with your streak logic

side_perf = (
    df2.groupby(gcols + ["side"])
       .agg(n=("acc", "size"), acc=("acc", "mean"))
       .reset_index()
)

# wide format (pos/neg columns)
side_wide = side_perf.pivot(index=gcols, columns="side", values=["acc", "n"])
side_wide


acc               n       
side                                         neg       pos   neg    pos
model         train_years feature_set                                  
random_forest 4           daily         0.557895  0.684211  95.0  133.0
                          daily+minute  0.284211  0.781955  95.0  133.0
                          minute        0.168421  0.774436  95.0  133.0
              6           daily         0.526316  0.744361  95.0  133.0
                          daily+minute  0.242105  0.827068  95.0  133.0
                          minute        0.147368  0.819549  95.0  133.0
xgboost       4           daily         0.526316  0.639098  95.0  133.0
                          daily+minute  0.368421  0.669173  95.0  133.0
                          minute        0.231579  0.601504  95.0  133.0
              6           daily         0.515789  0.714286  95.0  133.0
                          daily+minute  0.421053  0.654135  95.0  133.0
                          minute        0.315789  0.669173  95.0  133.0

In [55]:
K = 3
gcols = ["model", "train_years", "feature_set"]

d = df.copy()

# keep exact -3..+3; collapse only beyond into +/-4 (representing 3+)
d["streak_bucket"] = d["streak"].clip(lower=-K, upper=K)
d.loc[d["streak"] < -K, "streak_bucket"] = -(K + 1)   # strictly less than -3
d.loc[d["streak"] >  K, "streak_bucket"] =  (K + 1)   # strictly greater than +3

flip_perf = (
    d.groupby(gcols + ["streak_bucket"])
     .agg(n=("acc", "size"), acc=("acc", "mean"))
)

flip_wide = pd.concat(
    {"acc": flip_perf["acc"].unstack("streak_bucket"),
     "n":   flip_perf["n"].unstack("streak_bucket")},
    axis=1
)

# order: +1 acc, -1 acc, +1 n, -1 n, ... +3 acc, -3 acc, +3 n, -3 n, 3+ acc, -3+ acc, 3+ n, -3+ n
ordered_cols = []
for k in [1, 2, 3, "3+"]:
    pb = (K + 1) if k == "3+" else k
    nb = -(K + 1) if k == "3+" else -k
    ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]

flip_wide = flip_wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

# relabel buckets
rename_cols = []
for metric, b in flip_wide.columns:
    if b == (K + 1): lab = "3+"
    elif b == -(K + 1): lab = "-3+"
    else: lab = str(b)
    rename_cols.append((metric, lab))
flip_wide.columns = pd.MultiIndex.from_tuples(rename_cols)

flip_wide

acc             n           acc  \
                                               1        -1   1  -1         2   
model         train_years feature_set                                          
random_forest 4           daily         0.277778  0.147059  36  34  0.774194   
                          daily+minute  0.555556  0.058824  36  34  0.903226   
                          minute        0.666667  0.117647  36  34  0.677419   
              6           daily         0.388889  0.088235  36  34  0.838710   
                          daily+minute  0.555556  0.058824  36  34  0.935484   
                          minute        0.722222  0.088235  36  34  0.806452   
xgboost       4           daily         0.472222  0.147059  36  34  0.709677   
                          daily+minute  0.527778  0.176471  36  34  0.774194   
                          minute        0.583333  0.264706  36  34  0.612903   
              6           daily         0.472222  0.205882  36  34  0.645161   
                          daily+minute  0.611111  0.205882  36  34  0.709677   
                          minute        0.666667  0.352941  36  34  0.741935   

                                                   n       acc             n  \
                                              -2   2  -2     3        -3   3   
model         train_years feature_set                                          
random_forest 4           daily         0.791667  31  24  0.75  0.785714  20   
                          daily+minute  0.500000  31  24  0.90  0.428571  20   
                          minute        0.166667  31  24  0.85  0.214286  20   
              6           daily         0.708333  31  24  0.80  0.714286  20   
                          daily+minute  0.333333  31  24  0.90  0.214286  20   
                          minute        0.083333  31  24  0.90  0.285714  20   
xgboost       4           daily         0.750000  31  24  0.65  0.642857  20   
                          daily+minute  0.500000  31  24  0.75  0.428571  20   
                          minute        0.291667  31  24  0.65  0.142857  20   
              6           daily         0.708333  31  24  0.80  0.714286  20   
                          daily+minute  0.583333  31  24  0.65  0.642857  20   
                          minute        0.333333  31  24  0.65  0.214286  20   

                                                 acc             n      
                                        -3        3+       -3+  3+ -3+  
model         train_years feature_set                                   
random_forest 4           daily         14  0.913043  0.782609  46  23  
                          daily+minute  14  0.826087  0.304348  46  23  
                          minute        14  0.891304  0.217391  46  23  
              6           daily         14  0.934783  0.869565  46  23  
                          daily+minute  14  0.934783  0.434783  46  23  
                          minute        14  0.869565  0.217391  46  23  
xgboost       4           daily         14  0.717391  0.782609  46  23  
                          daily+minute  14  0.673913  0.478261  46  23  
                          minute        14  0.586957  0.173913  46  23  
              6           daily         14  0.913043  0.652174  46  23  
                          daily+minute  14  0.652174  0.434783  46  23  
                          minute        14  0.630435  0.304348  46  23